In [1]:
import findspark
findspark.init()

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# ZONA DE IMPORTS

In [4]:
from pyspark.sql.functions import split, col

# SPLIT  en pyspark

## Ejecicio 1


In [6]:
data = ["1,Jose\t30|Salamanca"]
df = spark.createDataFrame(data, "string")

In [12]:
df.show()

+--------------------+
|               value|
+--------------------+
|1,Jose\t30|Salamanca|
+--------------------+



In [13]:
split_col = split(df.value, ',|\t|\|')

In [34]:
print(split_col.getItem(2))

Column<'split(value, ,|	|\|, -1)[2]'>


In [16]:
df = df.withColumn('id', split_col.getItem(0))\
        .withColumn('name', split_col.getItem(1))\
        .withColumn('age', split_col.getItem(2))\
        .withColumn('city', split_col.getItem(3))

In [17]:
df_select = df.select('id','name', 'age', 'city')
df_select.show()

+---+----+---+---------+
| id|name|age|     city|
+---+----+---+---------+
|  1|Jose| 30|Salamanca|
+---+----+---+---------+



#EJEMPLO 2

Se tiene que hacer split por ":"

In [36]:
df_data = spark.createDataFrame([("1:a:2001",),("2:b:2002",),("3:c:2003",)],["value"])
df_data.show()

+--------+
|   value|
+--------+
|1:a:2001|
|2:b:2002|
|3:c:2003|
+--------+



In [44]:
#Se tiene que hacer el split por :, además de ir recoriendo y añadiendo las nuevas columnas
split_col = split(df_data.value, ":")
df_data_split = df_data.withColumn("id", split_col.getItem(0))\
                        .withColumn("letters", split_col.getItem(1))\
                        .withColumn("year", split_col.getItem(2))\
                        .drop("value")
df_data_split.show()

+---+-------+----+
| id|letters|year|
+---+-------+----+
|  1|      a|2001|
|  2|      b|2002|
|  3|      c|2003|
+---+-------+----+



In [43]:
#Otra forma de hacer el split y crear en una misma línea el dataframe con el schema, seria 
# con la funcion flatmap .
df_map = df_data.select(split(df_data.value, ":")).rdd.flatMap(lambda x: x).toDF(schema= ["id", "letters", "years"])
df_map.show()

+---+-------+-----+
| id|letters|years|
+---+-------+-----+
|  1|      a| 2001|
|  2|      b| 2002|
|  3|      c| 2003|
+---+-------+-----+

